# 🚗 VIDES Custom Vehicle ReID Training
**Step-by-step training script for OSNet-AIN.**

### Pre-requisites:
1. Upload `dataset.zip` to your Google Drive Root.
2. (Optional) Upload `osnet_pretrained.pth` to your Google Drive Root.
3. Set Runtime to **GPU** (Runtime > Change runtime type > T4 GPU).

In [ ]:
# STEP 1: PREPARE ENVIRONMENT
# ==========================================================================
from google.colab import drive
import os
import shutil
import glob

print("Connecting to Google Drive...")
drive.mount('/content/drive', force_remount=True)

print("Installing TorchReID...")
!git clone https://github.com/KaiyangZhou/deep-person-reid.git
%cd deep-person-reid
!pip install -q -r requirements.txt
!python setup.py develop
print("✅ Environment Ready.")

In [ ]:
# STEP 2: SETUP DATA & WEIGHTS
# ==========================================================================
print("Setting up Workspace...")
os.makedirs('/content/deep-person-reid/pretrained', exist_ok=True)
os.makedirs('configs', exist_ok=True)

# 1. Clean old data
if os.path.exists('/content/data'):
    shutil.rmtree('/content/data')

# 2. Unzip Dataset
zip_path = '/content/drive/MyDrive/dataset.zip'
if os.path.exists(zip_path):
    print("Unzipping dataset... (This may take a minute)")
    !unzip -q "$zip_path" -d /content/data
    print("✅ Dataset Unzipped.")
else:
    raise FileNotFoundError("❌ dataset.zip NOT FOUND in Google Drive! Please upload it to the main folder.")

# 3. Copy Weights
weights_path = '/content/drive/MyDrive/osnet_pretrained.pth'
dest_weights = '/content/deep-person-reid/pretrained/osnet_ain_x1_0_imagenet.pth'
if os.path.exists(weights_path):
    shutil.copy(weights_path, dest_weights)
    print("✅ Pre-trained weights loaded.")
else:
    print("⚠️ Pre-trained weights not found. Will download automatically (slower).")

In [ ]:
# STEP 3: CREATE CONFIGURATION
# ==========================================================================
config_content = """
model:
  name: 'osnet_ain_x1_0'
  pretrained: True

data:
  type: 'image'
  sources: ['market1501']
  targets: ['market1501']
  height: 256
  width: 128
  combineall: False
  transforms: ['random_flip']
  save_dir: 'log/my_model'

loss:
  name: 'softmax'
  softmax:
    label_smooth: True

train:
  optim: 'amsgrad'
  lr: 0.0015
  max_epoch: 60
  batch_size: 64
  fixbase_epoch: 0
  open_layers: ['classifier']

test:
  batch_size: 300
  dist_metric: 'euclidean'
  normalize_feature: False
  evaluate: True
  eval_freq: 10
  rerank: False
"""

with open('configs/my_custom_config.yaml', 'w') as f:
    f.write(config_content)
print("✅ Config file created.")

In [ ]:
# STEP 4: REGISTER DATASET (AUTO-DETECT PATH)
# ==========================================================================
import torchreid
from torchreid.data import ImageDataset

class CustomVehicleDataset(ImageDataset):
    def __init__(self, root='/content/data', **kwargs):
        # Auto-detect folder containing "bounding_box_train"
        print(f"Scanning {root} for dataset...")
        search_path = os.path.join(root, "**", "bounding_box_train")
        found = glob.glob(search_path, recursive=True)
        
        if not found:
            print(f"DEBUG: Directories in {root}:")
            for r, d, f in os.walk(root):
                for name in d: print(os.path.join(r, name))
            raise FileNotFoundError("❌ Error: Could not find 'bounding_box_train' folder inside dataset.zip")
            
        self.dataset_dir = os.path.dirname(found[0])
        print(f"✅ Dataset found at: {self.dataset_dir}")
        
        train_dir = os.path.join(self.dataset_dir, 'bounding_box_train')
        query_dir = os.path.join(self.dataset_dir, 'query')
        gallery_dir = os.path.join(self.dataset_dir, 'bounding_box_test')
        
        train = self.process_dir(train_dir)
        query = self.process_dir(query_dir)
        gallery = self.process_dir(gallery_dir)
        
        super(CustomVehicleDataset, self).__init__(train, query, gallery, **kwargs)

torchreid.data.register_image_dataset('my_custom_vehicle', CustomVehicleDataset)

In [ ]:
# STEP 5: RUN TRAINING
# ==========================================================================
print("🚀 STARTING TRAINING (OSNet-AIN)...")
print("Monitor the 'Rank-1' accuracy in the logs below.")

!python scripts/main.py \
    --config-file configs/my_custom_config.yaml \
    --root /content/data \
    dataset.name my_custom_vehicle \
    dataset.train_batch_size 64 \
    model.name osnet_ain_x1_0 \
    model.load_weights "/content/deep-person-reid/pretrained/osnet_ain_x1_0_imagenet.pth" \
    data.height 384 \
    data.width 384 \
    train.max_epoch 60 \
    train.lr 0.0015 \
    train.save_dir "log/my_model" \
    test.evaluate True \
    test.eval_freq 10

In [ ]:
# STEP 6: SAVE MODEL
# ==========================================================================
print("💾 Saving model to Google Drive...")
output_path = "/content/drive/MyDrive/final_vehicle_reid_model.pth"

possible_files = [
    "log/my_model/model.pth.tar-60",
    "log/my_model/model.pth.tar-50", 
    "log/my_model/model-best.pth.tar" 
]

found = False
for p in possible_files:
    if os.path.exists(p):
        shutil.copy(p, output_path)
        print(f"✅ SUCCESS! Model saved to: {output_path}")
        found = True
        break

if not found:
    print("❌ ERROR: Model not found. Did training complete?")